# Lesson 03 - Πρότυπα Σχεδιασμού Πράκτορα

Σε αυτό το μάθημα, εξερευνούμε τρία θεμελιώδη πρότυπα σχεδιασμού για την κατασκευή αποτελεσματικών πρακτόρων AI:

1. **Σαφείς Οδηγίες Πράκτορα** — Δημιουργία ακριβών, καθοριστικών ρόλων προσδιοριστικών εντολών που καθοδηγούν τη συμπεριφορά του πράκτορα  
2. **Δομημένη Έξοδος με Μοντέλα Pydantic** — Διασφάλιση ότι οι πράκτορες επιστρέφουν προβλέψιμα, επικυρωμένα δεδομένα  
3. **Πράκτορες Μοναδικής Ευθύνης** — Σχεδιασμός εστιασμένων πρακτόρων που ο καθένας εκτελεί καλά ένα συγκεκριμένο έργο  

Θα εφαρμόσουμε κάθε πρότυπο σε ένα σενάριο **συστήματος σύστασης ταξιδιωτικού προορισμού**, κατασκευάζοντας σταδιακά ένα σύστημα που μπορεί να προτείνει προορισμούς, να ελέγχει τη διαθεσιμότητα και να διαχειρίζεται τη λογιστική.


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Μοτίβο 1: Σαφείς Οδηγίες για τον Πράκτορα

Το πιο αποτελεσματικό μοτίβο είναι επίσης το πιο απλό: η γραφή σαφών, λεπτομερών οδηγιών για τον πράκτορά σας.

Καλές οδηγίες ορίζουν:
- **Ποιος** είναι ο πράκτορας (προσωπικότητα και τόνος)
- **Τι** πρέπει να κάνει (ευθύνες βήμα προς βήμα)
- **Πώς** πρέπει να συμπεριφέρεται (περιορισμοί και στυλ)

Παρακάτω, δημιουργούμε έναν πράκτορα ταξιδιωτικού κονσιέρζ με ρητές οδηγίες που διαμορφώνουν κάθε απάντηση που παράγει.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Πρότυπο 2: Δομημένη Έξοδος με Μοντέλα Pydantic

Το ελεύθερο κείμενο είναι χρήσιμο για τη συνομιλία, αλλά τα συστήματα στα κατώτερα επίπεδα χρειάζονται δομημένα δεδομένα.  
Συνδυάζοντας **μοντέλα Pydantic** με μια **συνάρτηση εργαλείου**, μπορούμε:

- Να ορίσουμε ένα ακριβές σχήμα για την έξοδο του πράκτορα  
- Να επικυρώνουμε τις απαντήσεις αυτόματα  
- Να ενσωματώσουμε αξιόπιστα τα αποτελέσματα του πράκτορα στην επιχειρησιακή λογική  

Εισάγουμε επίσης ένα εργαλείο που επιστρέφει λεπτομέρειες προορισμού ώστε ο πράκτορας να στηρίζει τις προτάσεις του σε πραγματικά δεδομένα.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Πρότυπο 3: Πράκτορες Μονής Ευθύνης

Οι σύνθετες εργασίες ωφελούνται από τη διανομή της δουλειάς σε πολλούς εστιασμένους πράκτορες, καθένας με μία μοναδική ευθύνη:

- Ένας **Ειδικός Προορισμού** που γνωρίζει για μέρη και διαθεσιμότητα
- Ένας **Σχεδιαστής Logistics** που διαχειρίζεται πτήσεις, ξενοδοχεία και δρομολόγια

Αυτό αντικατοπτρίζει την αρχή μηχανικής λογισμικού της *διαχωρισμού των ανησυχιών* — κάθε πράκτορας είναι πιο εύκολος στο να δοκιμαστεί, να διατηρηθεί και να βελτιωθεί ανεξάρτητα.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Σύνοψη

Σε αυτό το μάθημα εφαρμόσαμε τρία προτύπα σχεδίασης με πράκτορες σε ένα σενάριο συστήματος σύστασης ταξιδιών:

| Πρότυπο | Κύρια Ιδέα | Όφελος |
|---|---|---|
| **Σαφείς Οδηγίες** | Ορισμός persona, ευθυνών, και περιορισμών εκ των προτέρων | Συνεπής συμπεριφορά πράκτορα σύμφωνα με το brand |
| **Δομημένη Εξαγωγή** | Χρήση μοντέλων Pydantic ως μορφή απόκρισης | Ελεγμένα, κατάλληλα για μηχανική ανάγνωση αποτελέσματα |
| **Ενιαία Ευθύνη** | Ανάθεση μίας και μόνο εστιασμένης εργασίας σε κάθε πράκτορα | Εύκολο στο testing, συντήρηση, και σύνθεση |

Αυτά τα πρότυπα συνδυάζονται φυσικά — μπορείτε να συνδυάσετε σαφείς οδηγίες με δομημένη έξοδο μέσα σε έναν πράκτορα με ενιαία ευθύνη για να δημιουργήσετε στιβαρά, έτοιμα για παραγωγή συστήματα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία αυτόματης μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που προσπαθούμε για ακρίβεια, παρακαλούμε να λάβετε υπόψη ότι οι αυτόματες μεταφράσεις μπορεί να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για οποιεσδήποτε παρεξηγήσεις ή παρανοήσεις προκύψουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
